In [ ]:
"""
Script para dividir dataset de imágenes+transcripciones en validation y test (50/50).

Estructura esperada:
  carpeta_raiz/
    subcarpeta1/
      imagen1.jpg
      imagen2.png
      transcripciones.txt   ← nombre_imagen<TAB|ESPACIO>transcripcion
    subcarpeta2/
      ...

Produce:
  val.json   → { "subcarpeta/imagen.jpg": "transcripcion", ... }
  test.json  → { "subcarpeta/imagen.jpg": "transcripcion", ... }
"""

import os
import json
import argparse
from pathlib import Path
from sklearn.model_selection import train_test_split

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

def find_txt_file(folder: Path) -> Path | None:
    """Devuelve el primer .txt que encuentre dentro de la carpeta."""
    for f in folder.iterdir():
        if f.suffix.lower() == ".txt":
            return f
    return None

def parse_txt(txt_path: Path) -> dict[str, str]:
    """
    Parsea el archivo de transcripciones.
    Soporta separadores: tabulación, coma o espacio (primer token = nombre de imagen).
    """
    entries = {}
    with open(txt_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if not line.strip():
                continue

            # Intentar separar por tab primero, luego por coma, luego por espacio
            if "\t" in line:
                parts = line.split("\t", 1)
            elif "," in line:
                parts = line.split(",", 1)
            else:
                parts = line.split(" ", 1)

            if len(parts) != 2:
                print(f"  ⚠️  Línea ignorada (formato inesperado): {line!r}")
                continue

            img_name, transcription = parts[0].strip(), parts[1].strip()
            entries[img_name] = transcription

    return entries

def collect_all_entries(root: Path) -> dict[str, str]:
    """
    Recorre todas las subcarpetas, lee el txt y devuelve un dict global
    con paths relativos a root como claves.
    """
    all_entries: dict[str, str] = {}

    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue

        txt_file = find_txt_file(folder)
        if txt_file is None:
            print(f"⚠️  Sin .txt en: {folder.name} — se omite")
            continue

        entries = parse_txt(txt_file)
        if not entries:
            print(f"⚠️  .txt vacío en: {folder.name} — se omite")
            continue

        print(f"📂 {folder.name}  →  {len(entries)} entradas")

        for img_name, transcription in entries.items():
            # Construir path relativo: subcarpeta/imagen.ext
            relative_path = f"{folder.name}/{img_name}"
            all_entries[relative_path] = transcription

    return all_entries

def split_and_save(
    root: Path,
    output_dir: Path,
    val_name: str = "val.json",
    test_name: str = "test.json",
    seed: int = 42,
) -> None:
    print(f"\n🔍 Leyendo carpeta raíz: {root}\n")
    all_entries = collect_all_entries(root)

    if len(all_entries) < 2:
        raise ValueError("Se necesitan al menos 2 entradas para hacer el split.")

    keys = list(all_entries.keys())
    vals = [all_entries[k] for k in keys]

    val_keys, test_keys, val_vals, test_vals = train_test_split(
        keys, vals, test_size=0.5, random_state=seed
    )

    val_dict  = dict(zip(val_keys,  val_vals))
    test_dict = dict(zip(test_keys, test_vals))

    output_dir.mkdir(parents=True, exist_ok=True)
    val_path  = output_dir / val_name
    test_path = output_dir / test_name

    with open(val_path, "w", encoding="utf-8") as f:
        json.dump(val_dict, f, ensure_ascii=False, indent=2)

    with open(test_path, "w", encoding="utf-8") as f:
        json.dump(test_dict, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Split completado:")
    print(f"   val  → {len(val_dict):>5} entradas  →  {val_path}")
    print(f"   test → {len(test_dict):>5} entradas  →  {test_path}")

In [ ]:
split_and_save(Path('dataset1/datos_testing'), Path('dataset1/splits_test'))

In [ ]:
"""
Script para dividir un JSON en train, val y test.

Distribución: train 70%, val 15%, test 15%

Entrada:
  data.json → { "clave": "valor", ... }

Produce:
  train.json → { "clave": "valor", ... }
  val.json   → { "clave": "valor", ... }
  test.json  → { "clave": "valor", ... }
"""

import json
import argparse
from pathlib import Path
from sklearn.model_selection import train_test_split

def split_json(
    input_path: Path,
    output_dir: Path,
    train_ratio: float = 0.70,
    seed: int = 42,
) -> None:
    with open(input_path, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise TypeError("El JSON debe ser un objeto { clave: valor, ... }")

    if len(data) < 3:
        raise ValueError("Se necesitan al menos 3 entradas para hacer el split.")

    keys = list(data.keys())
    vals = [data[k] for k in keys]

    train_keys, temp_keys, train_vals, temp_vals = train_test_split(
        keys, vals,
        test_size=round(1 - train_ratio, 10),
        random_state=seed,
    )

    val_keys, test_keys, val_vals, test_vals = train_test_split(
        temp_keys, temp_vals,
        test_size=0.5,
        random_state=seed,
    )

    splits = {
        "train": dict(zip(train_keys, train_vals)),
        "val":   dict(zip(val_keys,   val_vals)),
        "test":  dict(zip(test_keys,  test_vals)),
    }

    output_dir.mkdir(parents=True, exist_ok=True)
    total = len(data)

    print(f"\n✅ Split completado ({total} entradas totales):")
    for name, split in splits.items():
        out_path = output_dir / f"{name}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(split, f, ensure_ascii=False, indent=2)
        print(f"   {name:<5} → {len(split):>5} entradas  ({len(split)/total:.0%})  →  {out_path}")

In [ ]:
split_json(Path('nuevo/cal_9/transcr.json'), Path('nuevo/cal_9'), train_ratio=0.6)

In [ ]:
split_json(Path('syntetic_dataset/original/labels.json'), Path('syntetic_dataset/split'), train_ratio=0.6)

In [ ]:
split_json(Path('final_real_dataset/trsncr.json'), Path('final_real_dataset/'), train_ratio=0.6)

In [ ]:
for i in range(1,19):
    if i == 2:
        continue
    split_json(Path(f'nuevo/cal_{i}/transcr.json'), Path(f'nuevo/cal_{i}'), train_ratio=0.6)

In [ ]:
import os
import shutil

def copiar_imagenes(origen, destino):
    os.makedirs(destino, exist_ok=True)
    
    imagenes = [f for f in os.listdir(origen) if f.lower().endswith('.png')]
    
    if not imagenes:
        print("No se encontraron imágenes .jpg en el directorio origen.")
        return

    copiadas = 0
    duplicadas = []

    for imagen in imagenes:
        destino_path = os.path.join(destino, imagen)
        
        if os.path.exists(destino_path):
            duplicadas.append(imagen)
            continue
        
        shutil.copy2(os.path.join(origen, imagen), destino_path)
        copiadas += 1

    print(f"\n✅ {copiadas} imágenes copiadas.")
    
    if duplicadas:
        print(f"⚠️  {len(duplicadas)} imágenes omitidas por nombre duplicado:")
        for nombre in duplicadas:
            print(f"   - {nombre}")

In [ ]:
import os
import shutil
import json

def copiar_imagenes(origen, destino, json_path):
    os.makedirs(destino, exist_ok=True)

    with open(json_path, 'r', encoding='utf-8') as f:
        keys_validas = set(json.load(f).keys())

    imagenes = [f for f in os.listdir(origen) 
                if f.lower().endswith('.jpg') and f in keys_validas]

    if not imagenes:
        print("No se encontraron imágenes que coincidan con las llaves del JSON.")
        return

    copiadas = 0
    duplicadas = []

    for imagen in imagenes:
        destino_path = os.path.join(destino, imagen)

        if os.path.exists(destino_path):
            duplicadas.append(imagen)
            continue

        shutil.copy2(os.path.join(origen, imagen), destino_path)
        copiadas += 1

    print(f"\n✅ {copiadas} imágenes copiadas.")
    print(f"ℹ️  {len(keys_validas) - copiadas - len(duplicadas)} llaves del JSON sin imagen en origen.")

    if duplicadas:
        print(f"⚠️  {len(duplicadas)} imágenes omitidas por nombre duplicado:")
        for nombre in duplicadas:
            print(f"   - {nombre}")

In [ ]:
# for i in range (0,39):
#     copiar_imagenes(f'syntetic_dataset/original', 'final_real_dataset/train')

copiar_imagenes('final_dataset', 'final_real_dataset/test', 'final_dataset/test.json')

In [ ]:
import os

def contar_imagenes(carpeta, total_esperado):
    imagenes = [f for f in os.listdir(carpeta) if f.lower().endswith('.jpg')]
    total = len(imagenes)
    
    print(f"Imágenes encontradas: {total}")
    
    if total == total_esperado:
        print(f"✅ Coincide con el total esperado ({total_esperado})")
    else:
        diff = total_esperado - total
        if diff > 0:
            print(f"❌ Faltan {diff} imágenes (esperadas {total_esperado})")
        else:
            print(f"⚠️  Hay {abs(diff)} imágenes de más (esperadas {total_esperado})")

# Uso
contar_imagenes('final_dataset', 1574)

In [ ]:
for i in range(1,19):
    if i == 2:
        continue
    copiar_imagenes(f'nuevo/cal_{i}', 'final_dataset')

In [ ]:
import json

def concatenar_json(x_archivo, y_archivo):
    """
    Concatena el JSON de x_archivo con y_archivo y guarda en y_archivo
    
    Args:
        x_archivo (str): Ruta del archivo JSON origen
        y_archivo (str): Ruta del archivo JSON destino
    """
    # Leer archivos
    with open(x_archivo, 'r', encoding='utf-8') as f:
        datos_x = json.load(f)
    
    with open(y_archivo, 'r', encoding='utf-8') as f:
        datos_y = json.load(f)
    
    # Verificar llaves duplicadas si ambos son dicts
    if isinstance(datos_x, dict) and isinstance(datos_y, dict):
        llaves_duplicadas = set(datos_x.keys()) & set(datos_y.keys())
        if llaves_duplicadas:
            print(f"⚠️  Advertencia: Las siguientes llaves existen en ambos archivos ({x_archivo} y {y_archivo}):")
            for llave in sorted(llaves_duplicadas):
                print(f"   - {llave}")
    
    # Lógica de concatenación
    if type(datos_x) == type(datos_y):
        if isinstance(datos_x, list):
            resultado = datos_x + datos_y
        elif isinstance(datos_x, dict):
            resultado = {**datos_y, **datos_x}  # x tiene prioridad
        else:
            resultado = [datos_x, datos_y]
    else:
        resultado = [datos_x, datos_y]
    
    # Guardar resultado
    with open(y_archivo, 'w', encoding='utf-8') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False)
    
    return resultado

In [ ]:
resultado = concatenar_json('final_dataset/val.json', 'final_real_dataset/val/val.json')

In [ ]:
# Uso
for i in range(1, 19):
    if i == 2:
        continue
    resultado = concatenar_json(f'nuevo/cal_{i}/train.json', 'nuevo/train.json')
    print(f"✅ Concatenación exitosa. Tipo: {type(resultado).__name__}")

In [ ]:
for i in range(0, 39):
    resultado = concatenar_json(f'original_docs/segm/img{i}/test.json', 'original_docs/segm/test.json')
    print(f"✅ Concatenación exitosa. Tipo: {type(resultado).__name__}")

In [ ]:
import json
import os

def renombrar_imagenes(carpeta, json_path, k):
    """
    Renombra las imágenes de una carpeta y actualiza sus llaves en el JSON,
    numerando ascendentemente desde k.

    Args:
        carpeta (str): Ruta de la carpeta con imágenes
        json_path (str): Ruta del archivo JSON
        k (int): Número inicial para renombrar
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        datos = json.load(f)

    # Obtener imágenes que están tanto en la carpeta como en el JSON
    archivos_carpeta = set(os.listdir(carpeta))
    imagenes = sorted([nombre for nombre in datos.keys() if nombre in archivos_carpeta])

    nuevo_datos = {}
    for i, nombre_original in enumerate(imagenes):
        extension = os.path.splitext(nombre_original)[1]
        nuevo_nombre = f"{k + i}{extension}"

        # Renombrar archivo
        ruta_original = os.path.join(carpeta, nombre_original)
        ruta_nueva = os.path.join(carpeta, nuevo_nombre)
        os.rename(ruta_original, ruta_nueva)

        # Actualizar llave en el JSON
        nuevo_datos[nuevo_nombre] = datos[nombre_original]

        print(f"  {nombre_original} → {nuevo_nombre}")

    # Guardar JSON actualizado
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(nuevo_datos, f, indent=2, ensure_ascii=False)

    print(f"\n✅ {len(imagenes)} imágenes renombradas desde {k} hasta {k + len(imagenes) - 1}")

renombrar_imagenes(Path('fotos_camara/segmentado2'), Path('fotos_camara/segmentado2/transcr.json'), 1540)